## ___Multi Response Phylogenetic Mixed Models `MR-PMM` using Markov Chain Monte Carlo generalized linear mixed models `MCMCglmm`___
----------------------

In [2]:
# multi response phylogenetic mixed effect models - look up https://benjamin-halliwell.github.io/MR-PMM/MR-PMM_euc_example_analysis.html

In [2]:
set.seed(2026 - 3 - 9)

suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("MCMCglmm")
    library("brms")
})

In [51]:
data <- read.csv("../../data/chapter2/FREDv3subset/collab_fineroots_log_995_species_means_5states_name_matched_with_phylogeny.csv") # log transformed SRL and RD species averages
phylogeny <- ape::read.tree("../../data/chapter2/uphylomaker/collab_fineroots_log_995_species_means_5states.tre") # phylogenetic tree
stopifnot(data$binominal==phylogeny$tip.label) # make sure the binominal names are matched between the trait data and the phylogeny

In [4]:
ape::is.binary(phylogeny) # damn

[1] FALSE

In [5]:
ape::is.ultrametric(phylogeny) # :)

[1] TRUE

In [6]:
phylogeny <- ape::multi2di(phylogeny) # make the phylogeny completely bifurcating by introducing 0 length branches
sum(phylogeny$edge.length == 0) # damn

[1] 91

In [7]:
phylogeny$edge.length[phylogeny$edge.length==0] <- rnorm(n = sum(phylogeny$edge.length == 0), mean = 1e-6, sd = 1e-8) # replace the 0 length edges with random noise
sum(phylogeny$edge.length == 0) # no more 0 length branches

[1] 0

In [8]:
# "By including more diverse species in our phylogeny, we capture deeper splits that represent more meaningful divergences in the genotype and phenotype of extant lineages,
# precisely the effects we intend to model when analysing inter-species data."
# "One consequence of this, is that shallow topology (near the tips) is less informative than deep topology when attempting to infer patterns of phylogenetic niche conservatism, because differences between genera
# are usually more significant than differences between species within genera."

In [9]:
# "For higher taxonomic ranks (e.g. genus), it will usually be possible to derive a unique consensus tree by sampling a single species from each genus and simply pruning off the other tips from the tree.
# This approach may be problematic for lower taxonomic ranks however, because more closely related species are less likely to be monophyletic with respect to the taxonomic rank in question.
# Even in such cases, we can easily account for this phylogenetic uncertainty by randomly sampling topologies at the specified rank and fitting our models over this sample of trees."

In [32]:
# harvest the genus names
gsub(phylogeny$tip.label, pattern = "_[a-z]+", replacement = '')[1:10]

[1] "Rudbeckia"  "Ratibida"   "Heliopsis"  "Liatris"    "Arnica"    
 [6] "Arnica"     "Hymenoxys"  "Helianthus" "Helianthus" "Helianthus"

In [44]:
setdiff(gsub(phylogeny$tip.label, pattern = "_[a-z]+", replacement = ''), unique(data$F01286)) # regex needs changes

[1] "Symphyotrichum-angliae" "Leucaena_Leucocephala"  "Festuca-bernardii"     
[4] "Laurelia-zelandiae"     "Blechnum-zelandiae"

In [48]:
data[data$F01286 == "Laurelia", ] # the hyphen is part of the specific epithet

,binominal,F01286,F01287,F01289,F01290,F00056,F00004,F00679,F00727,state
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>
896,Laurelia_novae-zelandiae,Laurelia,novae-zelandiae,Atherospermataceae,Laurales,NA,"Kramer-Walter KR, Bellingham PJ, Millar TR, Smissen RD, Richardson SJ, Laughlin DC. 2016. Root traits are multidimensional: specific root length is independent from root tissue density and the plant economic spectrum. Journal of Ecology 104: 1299-1310.",-0.04908521,2.371937,AM


In [50]:
data[data$F01286 == "Leucaena", ] # we have a specific epithet starting with a capital?????

,binominal,F01286,F01287,F01289,F01290,F00056,F00004,F00679,F00727,state
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>
343,Leucaena_Leucocephala,Leucaena,Leucocephala,Fabaceae,Fabales,NA,"Rajab YA, Holscher D, Leuschner C, Barus H, Tjoa A, Hertel D. 2018. Effects of shade tree cover and diversity on root system structure and dynamics in cacao agroforests: The role of root competition and space partitioning. Plant Soil 422: 349-369.",-0.6286087,3.02205,AM


In [49]:
setdiff(gsub(phylogeny$tip.label, pattern = "_[-a-z]+", replacement = '', fixed = FALSE), unique(data$F01286))

[1] "Leucaena_Leucocephala"

In [10]:
# sample the phylogeny repeatedly, with one randomly chosen species per genera

NUNIQUE_GENERA <- length(unique(data$F01286)) # number of unique genera in our phylogeny
NSAMPLES = 100 # number of times to sample the original phylogeny
sampled_phylogenies <- list() # sampled sub phylogeneies with genera at tips
sampled_vcvs <- list()

for (i in 1:NSAMPLES) {
    sampled_species <- mapply(split.data.frame(data[, c("binominal", "F01286")], ~F01286), FUN = function(df) sample(df$binominal, 1)) # this is a vector of one randomly sampled species per each genera in the phylogeny
    subphylogeny <- ape::keep.tip(phy = phylogeny, tip = unname(sampled_species), trim.internal = TRUE) # trimmed phylogeny with one randomly sampled species per genus
    stopifnot(length(subphylogeny$tip.label)==NUNIQUE_GENERA)
                                     
    if(!ape::is.binary(subphylogeny)) subphylogeny <- ape::multi2di(subphylogeny) # make bifucracting if not already
    if(!ape::is.ultrametric(subphylogeny)) subphylogeny <- phytools::force.ultrametric(subphylogeny, method = "extend", message = FALSE) # make ultrametric if not already

    subphylogeny$tip.label <- gsub(subphylogeny$tip.label, pattern = "_[a-z]+", replacement = '') # replace the tip labels (binominal names) with genus names
    
    sampled_phylogenies[[i]] <- subphylogeny
    sampled_vcvs[[i]] <- ape::vcv.phylo(phy = subphylogeny, corr = TRUE) # computes the expected variances and covariances of a continuous phenotype assuming it evolves under a Brownian motion model
}